In [ ]:
import os
import re
import numpy as np
import pandas as pd

# Rutas relativas respetando la estructura del proyecto
RAW_PATH = os.path.join("..", "data", "raw", "datasetairbnb.csv")  
PROCESSED_PATH = os.path.join("..", "data", "processed", "datasetairbnb_limpio.csv")

# Cargar dataset original
df_raw = pd.read_csv(RAW_PATH, low_memory=False)

print(f"Dimensiones iniciales: {df_raw.shape[0]} filas y {df_raw.shape[1]} columnas")

In [ ]:
def seleccionar_columnas_clave(df):
    columnas_interes = [
        # Identificadores y geografía
        "id", "name", "neighbourhood_cleansed", "latitude", "longitude",
        # Características físicas
        "property_type", "room_type", "accommodates", "bedrooms", "beds", "bathrooms_text",
        # Reglas operativas
        "minimum_nights", "maximum_nights", "availability_30", "availability_365",
        # Métricas de reputación
        "host_is_superhost", "number_of_reviews", "review_scores_rating", "reviews_per_month",
        # Métricas de negocio (Precios y Demanda)
        "price", "price_quote_price_per_night", "estimated_occupancy_l365d", "estimated_revenue_l365d"
    ]
    # Filtramos solo las que existan en el DataFrame para evitar errores
    cols_presentes = [c for c in columnas_interes if c in df.columns]
    return df[cols_presentes].copy()

df_subset = seleccionar_columnas_clave(df_raw)
print(f"Columnas seleccionadas: {df_subset.shape[1]}")

In [ ]:
def limpiar_precio(df):
    df = df.copy()
    
    # Si price es texto, quitamos '$', comas y espacios
    if df["price"].dtype == "object":
        df["price_clean"] = (
            df["price"]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df["price_clean"] = pd.to_numeric(df["price_clean"], errors="coerce")
    else:
        df["price_clean"] = df["price"]
        
    # Si 'price' venía nulo, intentamos rescatar el valor de 'price_quote_price_per_night'
    if "price_quote_price_per_night" in df.columns:
        df["price_clean"] = df["price_clean"].fillna(df["price_quote_price_per_night"])
        
    return df

df_limpio = limpiar_precio(df_subset)
print("Ejemplo de precios convertidos a número:")
print(df_limpio[["price", "price_clean"]].head())

In [ ]:
def extraer_banos(df):
    df = df.copy()
    
    def parse_bano(texto):
        if pd.isna(texto):
            return np.nan
        texto_str = str(texto).lower().strip()
        if "half" in texto_str:
            return 0.5
        # Busca el primer número entero o decimal
        match = re.search(r"(\d+(\.\d+)?)", texto_str)
        if match:
            return float(match.group(1))
        return np.nan

    df["bathrooms_clean"] = df["bathrooms_text"].apply(parse_bano)
    return df

df_limpio = extraer_banos(df_limpio)
print("Baños limpios extraídos:")
print(df_limpio[["bathrooms_text", "bathrooms_clean"]].head())

In [ ]:
def estandarizar_variables(df):
    df = df.copy()
    
    # Estandarizar superhost: 't' -> True, 'f' -> False
    if "host_is_superhost" in df.columns:
        df["host_is_superhost"] = df["host_is_superhost"].map({"t": True, "f": False})
        df["host_is_superhost"] = df["host_is_superhost"].fillna(False)
        
    # Limpiar espacios en blanco en la comuna
    if "neighbourhood_cleansed" in df.columns:
        df["neighbourhood_cleansed"] = df["neighbourhood_cleansed"].astype(str).str.strip()
        
    return df

df_limpio = estandarizar_variables(df_limpio)

In [ ]:
def tratar_nulos(df):
    df = df.copy()
    
    # 1. Eliminar filas sin precio válido
    df = df.dropna(subset=["price_clean"])
    
    # 2. Imputar bedrooms y beds según la mediana de accommodates
    df["bedrooms_clean"] = df.groupby("accommodates")["bedrooms"].transform(
        lambda x: x.fillna(x.median())
    )
    df["beds_clean"] = df.groupby("accommodates")["beds"].transform(
        lambda x: x.fillna(x.median())
    )
    
    # Si aún quedan nulos por grupos pequeños, rellenar con 1
    df["bedrooms_clean"] = df["bedrooms_clean"].fillna(1)
    df["beds_clean"] = df["beds_clean"].fillna(1)
    df["bathrooms_clean"] = df["bathrooms_clean"].fillna(1.0)
    
    return df

df_limpio = tratar_nulos(df_limpio)

In [ ]:
print(df_limpio["price_clean"].dtype)
df_limpio["price_clean"].head()

In [ ]:
df_limpio["price_clean"] = pd.to_numeric(
    df_limpio["price_clean"].astype(str).str.replace(r"[$,]", "", regex=True),
    errors="coerce"
)

print(df_limpio["price_clean"].dtype)        # debe decir float64
print(df_limpio["price_clean"].isna().sum()) # cuántos no se pudieron convertir

df_limpio = filtrar_outliers(df_limpio)

In [ ]:
def filtrar_outliers(df):
    df = df.copy()
    
    # Condición de tarifa sospechosa
    filtro_minimo = df["price_clean"] >= 8000
    
    # Mantener tarifas altas solo si son chalets de montaña en Lo Barnechea para muchos huéspedes
    condicion_lujo_montana = (df["neighbourhood_cleansed"] == "Lo Barnechea") & (df["accommodates"] >= 6)
    filtro_maximo = (df["price_clean"] <= 2000000) | condicion_lujo_montana
    
    # Filtrar extremos absurdos (como los 18 o 38 millones detectados)
    filtro_errores_extremos = df["price_clean"] < 15000000
    
    df_filtrado = df[filtro_minimo & filtro_maximo & filtro_errores_extremos].copy()
    
    filas_descartadas = len(df) - len(df_filtrado)
    print(f"Se eliminaron {filas_descartadas} registros con precios distorsionados/outliers.")
    
    return df_filtrado

df_limpio = filtrar_outliers(df_limpio)

In [ ]:
print(df_limpio["minimum_nights"].dtype)
print((df_limpio["minimum_nights"] < 1).sum())
print(df_limpio.loc[df_limpio["minimum_nights"] < 1, "minimum_nights"].value_counts())

In [ ]:
(df_limpio["minimum_nights"] < 1).sum()

In [ ]:
n = (df_limpio["minimum_nights"] < 1).sum()
print(f"{n} filas de {len(df_limpio)} ({n / len(df_limpio):.2%})")

In [ ]:
df_limpio.loc[df_limpio["minimum_nights"] < 1, "minimum_nights"].value_counts()

In [ ]:
df_limpio[df_limpio["minimum_nights"] < 1].head(10)

In [ ]:
print(df_limpio["minimum_nights"].isna().sum())
print(df_limpio["minimum_nights"].dtype)

In [ ]:
df_limpio = df_limpio.dropna(subset=["minimum_nights"]).copy()

In [ ]:
def validar_dataset(df):
    print("--- REPORTE DE VALIDACIÓN TÉCNICA ---")
    
    # 1. Comprobar que no haya nulos en variables clave
    nulos_precio = df["price_clean"].isna().sum()
    nulos_comuna = df["neighbourhood_cleansed"].isna().sum()
    assert nulos_precio == 0, "Error: Hay valores nulos en el precio."
    assert nulos_comuna == 0, "Error: Hay comunas nulas."
    print("✔ Validación de Nulos Críticos: PASADA (0 nulos)")
    
    # 2. Comprobar rangos de valores lógicos
    assert (df["price_clean"] > 0).all(), "Error: Existen precios menores o iguales a 0."
    assert (df["minimum_nights"] >= 1).all(), "Error: Noches mínimas menores a 1."
    if "estimated_occupancy_l365d" in df.columns:
        assert (df["estimated_occupancy_l365d"].between(0, 365)).all(), "Error: Ocupación fuera del rango 0-365 días."
    print("✔ Validación de Rangos Lógicos: PASADA")
    
    print(f"✔ Dataset final listo con {len(df)} registros limpios.")

# Ejecutar validación
validar_dataset(df_limpio)

# Guardar en data/processed/
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df_limpio.to_csv(PROCESSED_PATH, index=False)
print(f"Archivo guardado exitosamente en: {PROCESSED_PATH}")